# Scam Shield: fine-tune the text head on Kaggle (splits-v2)

Runs `src/finetune.py` from the private repo on a Kaggle GPU. This notebook only clones and runs; never edit code here (CLAUDE.md). The full run trains MuRIL twice, at max-len 128 and 256, on train (+ translations + augmentations), selects the best epoch on **val only**, and never tokenises or scores a test row (`finetune.py` asserts it; evaluation happens later on the Mac).

Before running:
- **Settings -> Accelerator:** **GPU T4 x2** (a P100 may have no kernels in Kaggle's torch; cell 4 fails early if so). **Internet:** on (clone + Hugging Face download).
- **Add-ons -> Secrets:** `GH_TOKEN`, a fine-grained GitHub token with read-only *Contents* access to `bored-psychic/scam-shield`, attached to this notebook.
- **Input:** the private dataset `scam-shield-data`, **latest version** ("splits-v2 (e4b2e78)"). Its four CSVs (`split_map.csv`, `messages.csv`, `messages_adv_test.csv`, `messages_aug.csv`) and `build_meta.json` (the mask version of the data; `finetune.py` refuses to train without it) sit at the dataset root. Cell 1 finds each by name, fails if any is missing or exists twice (e.g. a leftover `final/` folder or an old dataset also attached), and checks the `split_map.csv` SHA-256 against splits-v2. An older version has a different hash and stops there.
- First run with `SMOKE = True` in cell 1 (a few minutes: both lengths on 2,000 rows, checks GPU memory at 256), then set it to `False` and run all.

**Leave-one-source-out (LOSO) folds:** set `LOSO_SOURCES` in cell 1 (e.g. `["uci_sms_spam", "india_fraud_final", "scam_ham_india", "synth_india"]`) and, as the cell checks, `USE_AUG = False`, `MAX_LENS = [128]`. The notebook then runs one fold per source instead of the two full runs: MuRIL with exactly the settings of the shipped bundle (`finetune.py` refuses anything else), the rule `cluster-strict-v1` (every train and val row in a near-duplicate cluster that holds a row of the source is dropped: its own rows, the translations of its parents and other sources' near copies), `--no-save-model` (no ~1 GB checkpoints), and only that source's val + test rows scored into `predictions_loso.csv`. Run dirs are `muril_loso_<source>_<UTC>`. **Settings -> Accelerator must be a T4:** the shipped bundle trained in fp16 on a T4, and cell 4 stops a LOSO run on any other GPU (`eval_loso.py` would refuse the folds anyway unless run with `--allow-precision-mismatch`). `SMOKE = True` runs the first source only, on 2,000 rows. Evaluation: `.venv/bin/python src/eval_loso.py` on the Mac.

Precision is picked by `finetune.py --precision auto`: T4 (7.5) and P100 (6.0) get fp16 + GradScaler; bf16 needs compute capability >= 8.0.

In [ ]:
# ---- settings ----------------------------------------------------------------------------------------------------
SMOKE = False                       # True: 1 epoch on 2,000 rows per config, to check GPU memory and the paths; then False
USE_AUG = True                      # also train on messages_aug.csv (train parents only; all scam rows)
MAX_LENS = [128, 256]               # one full run each, same batch / lr / epochs / seed, so only the token cut differs
LOSO_SOURCES = []                   # [] = the runs above. Else one leave-one-source-out fold per source, e.g. ["uci_sms_spam", "india_fraud_final", "scam_ham_india", "synth_india"];
                                    # then USE_AUG must be False and MAX_LENS [128] (a fold trains like the shipped bundle, only the source differs)
EXPECTED_SPLIT_MAP_SHA256 = "8263df1c304de7154290f57f648387ddc396cab303630753aa40200619bd4e6c"   # splits-v2; same constant as finetune.py
EXPECTED_MASK_VERSION = "v2"        # the pii.py mask set of the data (build_meta.json "mask_version"); finetune.py refuses data without one and records it in shield_meta.json
# splits-v3 (filled on 2 Nov, not before): the two EXPECTED_ lines above become
#   EXPECTED_SPLIT_MAP_SHA256 = "<TODO 2 Nov: SHA-256 of the splits-v3 split_map.csv, the same value as finetune.SPLIT_MAP_SHA256 and CLAUDE.md>"
#   EXPECTED_MASK_VERSION = "v3"
# and the new Kaggle dataset version must also carry build_meta.json (five files at the root now, not four).

import glob, hashlib, os
from pathlib import Path

# find each data file by name anywhere under /kaggle/input: exactly one copy of each, all in the same folder
NAMES = ["split_map.csv", "messages.csv", "messages_adv_test.csv", "messages_aug.csv", "build_meta.json"]
found = {n: sorted(glob.glob(f"/kaggle/input/**/{n}", recursive=True)) for n in NAMES}
problems = [f"{n}: {len(h)} copies {h}" for n, h in found.items() if len(h) != 1]
listing = "\n  ".join(sorted(os.listdir("/kaggle/input"))) or "(empty)"
if problems:
    raise RuntimeError("expected exactly one copy of each data file under /kaggle/input:\n  " + "\n  ".join(problems)
                       + f"\n/kaggle/input contains:\n  {listing}\nAttach only the latest scam-shield-data version (CSVs at its root, no final/ folder).")
folders = {str(Path(h[0]).parent) for h in found.values()}
if len(folders) != 1:
    raise RuntimeError(f"data files are in different folders: {folders}")
DATA_DIR = folders.pop()
h = hashlib.sha256()
with open(f"{DATA_DIR}/split_map.csv", "rb") as f:
    for block in iter(lambda: f.read(1 << 20), b""):
        h.update(block)
assert h.hexdigest() == EXPECTED_SPLIT_MAP_SHA256, (
    f"split_map.csv SHA-256 is {h.hexdigest()}, expected {EXPECTED_SPLIT_MAP_SHA256} (splits-v2): the notebook is attached to an old dataset version")
print("DATA_DIR =", DATA_DIR)
print("split_map.csv SHA-256 matches splits-v2:", h.hexdigest())
import json
build_meta = json.loads(Path(DATA_DIR, "build_meta.json").read_text(encoding="utf-8"))
assert build_meta.get("mask_version") == EXPECTED_MASK_VERSION, (
    f"build_meta.json mask_version is {build_meta.get('mask_version')!r}, expected {EXPECTED_MASK_VERSION!r}: the dataset version and this notebook do not match")
print("build_meta.json mask_version:", build_meta["mask_version"])
for n in NAMES:
    print(f"  {n}: {Path(DATA_DIR, n).stat().st_size / 1e6:.1f} MB")

if LOSO_SOURCES:
    assert not USE_AUG and MAX_LENS == [128], "LOSO folds train like the shipped bundle: set USE_AUG = False and MAX_LENS = [128] (finetune.py would refuse otherwise)"
    folds = LOSO_SOURCES[:1] if SMOKE else LOSO_SOURCES   # smoke: one fold on 2,000 rows
    RUNS = [f"--model muril --max-len 128 --exclude-source {s} --no-save-model" + (" --smoke" if SMOKE else "") for s in folds]
else:
    RUNS = [f"--model muril --max-len {L}" + (" --aug" if USE_AUG else "") + (" --smoke" if SMOKE else "") for L in MAX_LENS]
REPO_URL = "https://github.com/bored-psychic/scam-shield.git"
REPO_DIR = "/kaggle/working/scam-shield"
OUT_ROOT = "/kaggle/working/models"
print("runs:", *RUNS, sep="\n  ")

In [ ]:
# ---- clone (or update) the private repo; the token is never printed, never in argv, never written to .git/config -------
import base64, os, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

def git(*args, token=None, cwd=None):
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    if token:   # one-off config from the environment (git >= 2.31): an Authorization header for github.com, this call only
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")
    r = subprocess.run(["git", *args], cwd=cwd, env=env, capture_output=True, text=True)
    out = (r.stdout + r.stderr)
    if token:
        out = out.replace(token, "***").replace(basic, "***")
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out.strip()

token = UserSecretsClient().get_secret("GH_TOKEN")
try:
    if (Path(REPO_DIR) / ".git").exists():
        print(git("pull", "--ff-only", token=token, cwd=REPO_DIR))
    else:
        print(git("clone", "--quiet", REPO_URL, REPO_DIR, token=token) or f"cloned into {REPO_DIR}")
finally:
    del token

git("remote", "set-url", "origin", REPO_URL, cwd=REPO_DIR)   # plain https URL, no credentials
cfg = (Path(REPO_DIR) / ".git" / "config").read_text()
assert "@github.com" not in cfg and "extraheader" not in cfg.lower() and "x-access-token" not in cfg, "credentials in .git/config"
print("HEAD", git("rev-parse", "HEAD", cwd=REPO_DIR), "|", git("log", "-1", "--format=%s", cwd=REPO_DIR))
print("tree clean:", git("status", "--porcelain", "--untracked-files=no", cwd=REPO_DIR) == "")

In [ ]:
# ---- install only what's missing; keep Kaggle's own torch (the cu130 pin is for the RTX 5050 laptop) --------------------
import importlib.metadata as md, re, subprocess, sys

pins = {}
for line in open(f"{REPO_DIR}/requirements-finetune.txt"):
    m = re.match(r"^([A-Za-z0-9_.-]+)==(\S+)", line.strip())
    if m and m.group(1).lower() != "torch":
        pins[m.group(1)] = m.group(2)

missing, differ = [], []
for name, want in pins.items():
    try:
        have = md.version(name)
    except md.PackageNotFoundError:
        missing.append(f"{name}=={want}")
        continue
    if have != want:
        differ.append((name, have, want))
# tldextract is the exception: textprep.model_text uses its bundled suffix list, so its version changes model inputs (rule 11)
for name, have, want in differ:
    if name == "tldextract":
        missing.append(f"{name}=={want}")
if missing:
    print("installing:", " ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("nothing missing")
for name, have, want in differ:
    if name != "tldextract":
        print(f"kept Kaggle's {name} {have} (pin {want}); shield_meta.json records the version used")

In [ ]:
# ---- GPU and versions -----------------------------------------------------------------------------------------------
import subprocess, torch, transformers, tokenizers, tldextract
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| transformers", transformers.__version__,
      "| tokenizers", tokenizers.__version__, "| tldextract", tldextract.__version__)
assert torch.cuda.is_available(), "no GPU: Settings -> Accelerator"
if LOSO_SOURCES:   # the shipped bundle was trained in fp16 on a T4; a LOSO fold on another GPU would not be comparable (eval_loso.py checks the precision again)
    assert "T4" in torch.cuda.get_device_name(0), f"LOSO folds need a Kaggle T4 (fp16, like the shipped bundle); this GPU is {torch.cuda.get_device_name(0)}"
cap = torch.cuda.get_device_capability(0)
print("GPU", torch.cuda.get_device_name(0), "| compute capability", f"{cap[0]}.{cap[1]}",
      "| auto precision ->", "bf16" if cap >= (8, 0) else "fp16 + GradScaler")
print("torch built for:", torch.cuda.get_arch_list())
# fail here, not 10 minutes in, if this torch build has no kernels for the GPU (recent wheels dropped Pascal / P100, sm_60)
x = torch.randn(256, 256, device="cuda")
with torch.autocast("cuda", dtype=torch.float16):
    y = x @ x
torch.cuda.synchronize()
print("fp16 matmul ok:", tuple(y.shape), y.dtype)

In [ ]:
# ---- train: one run per entry in RUNS, one after the other ---------------------------------------------------------------
import re, shlex, subprocess, sys
from datetime import datetime, timezone

# the split_map hash finetune.py enforces must be the one this notebook just checked
pinned = re.search(r'^SPLIT_MAP_SHA256 = "([0-9a-f]{64})"', open(f"{REPO_DIR}/src/finetune.py", encoding="utf-8").read(), re.M).group(1)
assert pinned == EXPECTED_SPLIT_MAP_SHA256, f"finetune.py pins {pinned[:12]}..., the notebook expects {EXPECTED_SPLIT_MAP_SHA256[:12]}...: the clone is not the splits-v2 commit"

RUN_DIRS = []
for args in RUNS:
    parts = shlex.split(args)
    tag = f"len{parts[parts.index('--max-len') + 1]}" if "--max-len" in parts else "len128"
    if "--exclude-source" in parts:   # a LOSO fold: muril_loso_<source>_<UTC>
        tag = f"loso_{parts[parts.index('--exclude-source') + 1]}"
    run_dir = f"{OUT_ROOT}/muril_{tag}_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}{'_smoke' if SMOKE else ''}"
    cmd = [sys.executable, "-u", "src/finetune.py", *parts, "--data-dir", DATA_DIR, "--out", run_dir]
    print("\n" + " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    assert p.wait() == 0, f"finetune.py exited with {p.returncode} for: {args}"
    RUN_DIRS.append(run_dir)

In [ ]:
# ---- zip each run for download (without the epoch_* checkpoints; model/ is the best epoch), then free the checkpoints ------
import json, shutil, zipfile
from pathlib import Path

for rd in RUN_DIRS:
    run = Path(rd)
    zip_path = Path("/kaggle/working") / f"{run.name}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(run.rglob("*")):
            rel = f.relative_to(run)
            if f.is_file() and not rel.parts[0].startswith("epoch_"):
                z.write(f, Path(run.name) / rel)
    meta = json.loads((run / "shield_meta.json").read_text())
    print(f"\n{zip_path} ({zip_path.stat().st_size / 1e6:.0f} MB)")
    print({k: meta[k] for k in ("git_commit", "gpu_name", "precision", "best_epoch", "training_seconds", "threshold_suspicious", "threshold_likely")})
    print("max_len", meta["settings"]["max_len"], "| aug", meta["settings"]["aug"], "| rows", meta["rows"], "| val overall", meta["val_overall"])
    if meta.get("loso"):
        print("LOSO fold: held-out source", meta["exclude_source"], "| dropped", meta["loso_dropped"], "| held-out rows scored", meta["loso_held_out_rows"])
        assert sorted(p.name for p in run.glob("predictions_*")) == ["predictions_loso.csv", "predictions_val.csv"], "unexpected prediction files in a fold"
    else:
        print("test files written:", sorted(p.name for p in run.glob("predictions_*") if "val" not in p.name) or "none (as intended)")
    for ck in run.glob("epoch_*"):   # keep the notebook's saved output small; the zip and model/ hold the best epoch
        shutil.rmtree(ck)
print("\nDownload the zips from the Output panel (or Save Version -> Output), unzip into models/finetune/ on the Mac, and bring back each train_log.txt.")